# Process Persuade 2.0 Dataset in Google Colab

This notebook processes the full Persuade 2.0 dataset (25,996 essays) and exports it to JSON format.

**Estimated time**: 10-30 minutes


## Step 1: Install Dependencies


In [ ]:
!pip install pandas spacy
!python -m spacy download en_core_web_md


## Step 2: Upload Dataset CSV


In [ ]:
from google.colab import files
import os

print("Please upload your persuade_2.0_human_scores_demo_id_github.csv file:")
uploaded = files.upload()

# Get the filename
csv_filename = list(uploaded.keys())[0]
print(f"\n✓ Uploaded: {csv_filename} ({os.path.getsize(csv_filename) / (1024*1024):.1f} MB)")


## Step 3: Process Full Dataset (All-in-One Code)


In [ ]:
import pandas as pd
import json
import re
import time
import os

print("=" * 60)
print("Processing Persuade 2.0 Dataset")
print("=" * 60)

# Load CSV
print(f"\nStep 1/3: Loading {csv_filename}...")
df = pd.read_csv(csv_filename)
print(f"✓ Loaded {len(df):,} essays")

# Find text column
text_col = 'full_text' if 'full_text' in df.columns else None
if text_col is None:
    for col in df.columns:
        if df[col].dtype == 'object':
            sample = df[col].dropna().iloc[0] if len(df[col].dropna()) > 0 else ""
            if isinstance(sample, str) and len(sample) > 100:
                text_col = col
                break

if text_col is None:
    raise ValueError("Could not find text column")

print(f"Using text column: {text_col}")

# Show statistics
if 'word_count' in df.columns:
    print(f"\nWord count - Mean: {df['word_count'].mean():.1f}, Range: {df['word_count'].min()}-{df['word_count'].max()}")
if 'holistic_essay_score' in df.columns:
    scores = df['holistic_essay_score'].dropna()
    print(f"Score - Mean: {scores.mean():.2f}, Range: {scores.min():.2f}-{scores.max():.2f}")

# Filter by word count
print(f"\nStep 2/3: Filtering essays (150-1000 words)...")
df['word_count'] = df[text_col].apply(lambda x: len(str(x).split()) if pd.notna(x) else 0)
initial_count = len(df)
df = df[(df['word_count'] >= 150) & (df['word_count'] <= 1000)]
filtered = initial_count - len(df)
print(f"✓ {len(df):,} essays after filtering (removed {filtered:,} outside range)")

# Convert to standard format
print(f"\nStep 3/3: Converting to JSON format...")
essays = []
start_time = time.time()

for idx, row in df.iterrows():
    try:
        essay_id = str(row.get('essay_id_comp', f"essay_{idx}"))
        
        essay = {
            "id": essay_id,
            "text": str(row[text_col]).strip(),
            "teacher_annotations": [],
            "essay_type": "argumentative",
            "grade_level": str(row.get('grade_level', 'high_school')) if pd.notna(row.get('grade_level')) else 'high_school',
            "prompt": str(row.get('prompt_name', '')) if 'prompt_name' in row.index else '',
            "metadata": {
                "word_count": int(row['word_count']),
                "source": "persuade_2.0",
                "original_id": essay_id
            }
        }
        
        if 'holistic_essay_score' in row.index and pd.notna(row['holistic_essay_score']):
            essay["metadata"]["human_score"] = float(row['holistic_essay_score'])
        
        if 'discourse_type_num' in row.index and pd.notna(row['discourse_type_num']):
            essay["metadata"]["discourse_type"] = int(row['discourse_type_num'])
        
        essays.append(essay)
        
        # Progress update
        if (idx + 1) % 1000 == 0:
            elapsed = time.time() - start_time
            rate = (idx + 1) / elapsed
            remaining = (len(df) - idx - 1) / rate if rate > 0 else 0
            print(f"  Processed {idx + 1:,}/{len(df):,} ({((idx+1)/len(df)*100):.1f}%) - ETA: {remaining/60:.1f} min")
    except Exception as e:
        print(f"Warning: Error processing row {idx}: {e}")
        continue

# Save JSON
output_file = 'processed_persuade_full.json'
print(f"\nSaving to {output_file}...")
with open(output_file, 'w', encoding='utf-8') as f:
    json.dump(essays, f, indent=2, ensure_ascii=False)

file_size = os.path.getsize(output_file) / (1024*1024)
print(f"✓ Saved {len(essays):,} essays ({file_size:.1f} MB)")
print("\n" + "=" * 60)
print("✅ Processing Complete!")
print("=" * 60)


## Step 4: Download Processed JSON


In [ ]:
from google.colab import files

file_size = os.path.getsize(output_file) / (1024*1024)
print(f"Downloading {output_file} ({file_size:.1f} MB)...")
files.download(output_file)
print("\n✓ Download complete!")
print(f"\nNext steps:")
print(f"  1. Save file to: backend/data/processed_persuade_full.json")
print(f"  2. Use it in your EduCompose analysis pipeline")
